# 03 - Fine-tune DistilBERT cho Formality Classification

| | Chi tiet |
|---|---|
| **Train** | `gyafc_transformer_ready.csv` — **103,221 cau** (100% training data, KHONG chia 80/20) |
| **Test**  | `gyafc_test_ready.csv` — **2,664 cau** (GYAFC Official Test Set) |
| **Model** | `distilbert-base-uncased` |
| **Output**| `models/best_distilbert_formality/` |

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn -q

In [ ]:
# ============================================================
# Cell 1: Import & kiem tra GPU
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

if torch.cuda.is_available():
    print(f"[OK] GPU san sang: {torch.cuda.get_device_name(0)}")
else:
    print("[WARN] Khong co GPU - dung CPU")

In [ ]:
# Cell 2: Duong dan
BASE_DIR   = "c:/Users/Admin/Documents/testvscode/formalityClassificationProject"

# --- Tap TRAIN: 100% du lieu train, 
TRAIN_PATH = f"{BASE_DIR}/data/gyafc_transformer_ready.csv"

# --- Tap TEST: GYAFC Official Test Set ---
TEST_PATH  = f"{BASE_DIR}/data/gyafc_test_ready.csv"

# --- Checkpoint trung gian (Trainer tu luu theo epoch) ---
CKPT_DIR   = f"{BASE_DIR}/models/distilbert_formality_model"

# --- Thu muc luu model cuoi cung ---
SAVE_DIR   = f"{BASE_DIR}/models/best_distilbert_formality"

MODEL_NAME = "distilbert-base-uncased"

print(f"TRAIN_PATH : {TRAIN_PATH}")
print(f"TEST_PATH  : {TEST_PATH}")
print(f"SAVE_DIR   : {SAVE_DIR}")

In [ ]:
# Cell 3: Nap du lieu
print("Dang nap du lieu...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"  Train  : {len(train_df):,} cau  ")
print(f"  Test   : {len(test_df):,}  cau  ")
print(f"  Phan phoi nhan (train): {train_df['label'].value_counts().to_dict()}")
print(f"  Phan phoi nhan (test) : {test_df['label'].value_counts().to_dict()}")

# Chuyen sang Hugging Face Dataset
train_hg = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
test_hg  = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

In [ ]:
# Cell 4: Tokenization
print("Dang tokenize...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

train_tok = train_hg.map(tokenize_function, batched=True)
test_tok  = test_hg.map(tokenize_function,  batched=True)
print("Tokenization hoan tat!")

In [ ]:
# Cell 5: Khoi tao Model & Metrics
print("Khoi tao model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

metric_acc = evaluate.load("accuracy")
metric_f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)
    f1  = metric_f1.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

print("Model san sang!")

In [ ]:
# Cell 6: Huan luyen
print("Bat dau huan luyen DistilBERT...")

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",          # chon model tot nhat theo F1
    greater_is_better=True,
    report_to="none",
    logging_steps=500,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,   # 100% gyafc_transformer_ready.csv
    eval_dataset=test_tok,     # GYAFC Official Test Set
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 7: Luu model tot nhat
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)         # luu weights + config
tokenizer.save_pretrained(SAVE_DIR)  # luu tokenizer

# Kiem tra cac file da luu
saved = os.listdir(SAVE_DIR)
print(f"Files trong {SAVE_DIR}:")
for f in sorted(saved):
    mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f"  {f:45s}  {mb:.1f} MB")

has_weights = any(f in saved for f in ["model.safetensors", "pytorch_model.bin"])
if has_weights:
    print("\n[THANH CONG] Model da duoc luu day du!")
    print("Ban co the chay notebook 05_Evaluation.ipynb.")
else:
    print("\n[LOI] Khong tim thay file weights!")